# End-to-End Investment Banking 3-Statement Model & DCF Valuation

This educational notebook constructs a professional, Wall Street-grade financial model from scratch. It connects live market data to a highly structured Excel architecture.

## Step 1: Read Data & Chart of Accounts
### Why are we isolating specific rows?
Public companies report hundreds of highly specific accounting sub-ledgers in their SEC filings. Trying to forecast all 100+ rows is chaotic and unnecessary.

In investment banking, we map the raw data into a clean, simplified **Chart of Accounts** (`is_rows`, `bs_rows`, `cf_rows`). 
*   We capture the core operating metrics that actually drive the business.
*   All non-core accounting items we excluded are aggregated mathematically into a single **Historical Plug** on the Balance Sheet. This ensures the foundational accounting equation ($Assets = Liabilities + Equity$) remains perfectly balanced.

In [ ]:
# Install required libraries if running in a fresh Colab environment
!pip install yfinance openpyxl pandas

import yfinance as yf
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from google.colab import files

# ==========================================
# DATA INGESTION & CAPM VARIABLES
# ==========================================
print("Fetching TSCO data and market rates from yfinance...")
tsco = yf.Ticker("TSCO")
raw_is = tsco.financials
raw_bs = tsco.balance_sheet
raw_cf = tsco.cashflow
live_price = tsco.history(period="1d")["Close"].iloc[0]

# CAPM Market Data
tnx = yf.Ticker("^TNX")
risk_free_rate = tnx.history(period="1d")["Close"].iloc[0] / 100

# ==========================================
# MASTER ASSUMPTIONS (Model Inputs)
# ==========================================
assump_rev_growth = 0.05       # 5.0% Revenue Growth
assump_gross_margin = 0.35     # 35.0% Gross Margin
assump_sga_margin = 0.23       # 23.0% SG&A Margin
assump_tax_rate = 0.24         # 24.0% Tax Rate
assump_capex_margin = -0.04    # -4.0% of Revenue for CapEx
assump_shares_out = tsco.info.get("sharesOutstanding", 530000000) # Dynamic fetch to handle stock splits

# DCF Specific Assumptions
assump_beta = 1.05             # Damodaran Industry Proxy for Specialty Retail (overrides noisy yfinance beta)
assump_erp = 0.055             # 5.5% Equity Risk Premium
assump_tgr = 0.02              # 2.0% Terminal Growth Rate
assump_exit_mult = 14.0        # 14.0x EV/EBITDA Exit Multiple

# The Clean Chart of Accounts
is_rows = ["Total Revenue", "Cost Of Revenue", "Gross Profit", "Selling General And Administration", 
           "Operating Income", "Interest Expense", "Tax Provision", "Net Income"]
bs_rows = ["Cash And Cash Equivalents", "Receivables", "Inventory", "Net PPE", 
           "Accounts Payable", "Current Accrued Expenses", "Long Term Debt", 
           "Common Stock", "Retained Earnings", "Treasury Stock"]
cf_rows = ["Net Income From Continuing Operations", "Depreciation And Amortization", "Change In Working Capital", 
           "Operating Cash Flow", "Capital Expenditure", "Investing Cash Flow", 
           "Repayment Of Debt", "Cash Dividends Paid", "Repurchase Of Capital Stock", 
           "Financing Cash Flow", "Changes In Cash"]

def clean_yfinance_df(raw_df, target_rows):
    available_rows = [row for row in target_rows if row in raw_df.index]
    return raw_df.loc[available_rows].iloc[:, ::-1] 

clean_is = clean_yfinance_df(raw_is, is_rows)
clean_bs = clean_yfinance_df(raw_bs, bs_rows)
clean_cf = clean_yfinance_df(raw_cf, cf_rows)
print("Data extracted and mapped.")

# ==========================================
# WORKBOOK INITIALIZATION & STYLES
# ==========================================
wb = Workbook()
bold_font = Font(bold=True)
blue_font = Font(color="0000FF")   # Hardcoded Inputs
black_font = Font(color="000000")  # Standard Formulas
green_font = Font(color="008000")  # Cross-Sheet Links
title_font = Font(bold=True, size=14)
yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
green_fill = PatternFill(start_color="CCFFCC", end_color="CCFFCC", fill_type="solid")
blue_fill = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid")

forecast_cols = ["2026E", "2027E", "2028E", "2029E", "2030E"]

def write_clean_df(ws, df):
    ws.cell(row=1, column=1, value="Line Item").font = bold_font
    for c_idx, col_name in enumerate(df.columns, 2):
        col_header = col_name.strftime('%Y') if isinstance(col_name, pd.Timestamp) else str(col_name)[:4]
        ws.cell(row=1, column=c_idx, value=col_header).font = bold_font
    for r_idx, (index, row) in enumerate(df.iterrows(), 2):
        ws.cell(row=r_idx, column=1, value=index)
        for c_idx, val in enumerate(row, 2):
            cell = ws.cell(row=r_idx, column=c_idx, value=val)
            if pd.notnull(val): cell.font = blue_font

## Step 2: The Income Statement

The Income Statement projects the core profitability of the business. 

### Formulas Used:
*   **Total Revenue:** Prior Year Revenue * (1 + Revenue Growth Rate)
*   **Cost of Revenue:** Total Revenue * (1 - Gross Margin)
*   **Gross Profit:** Total Revenue - Cost of Revenue
*   **SG&A:** Total Revenue * SG&A Margin
*   **Operating Income (EBIT):** Gross Profit - SG&A
*   **Interest Expense:** Beginning Debt Balance * Interest Rate (calculated on the base amount to avoid circular dependencies)
*   **Tax Provision:** (Operating Income - Interest Expense) * Effective Tax Rate
*   **Net Income:** Operating Income - Interest Expense - Tax Provision

In [ ]:
ws_is = wb.create_sheet("Income Statement")
write_clean_df(ws_is, clean_is)
for i, year in enumerate(forecast_cols, start=7): ws_is.cell(row=1, column=i, value=year).font = bold_font

ws_is["A11"] = "ASSUMPTIONS"; ws_is["A11"].font = bold_font
ws_is["A12"] = "Revenue Growth (%)"
ws_is["A13"] = "Gross Margin (%)"
ws_is["A14"] = "SG&A Margin (%)"
ws_is["A15"] = "Effective Tax Rate (%)"

for col_idx in range(7, 12):
    col = get_column_letter(col_idx)
    prev = get_column_letter(col_idx - 1)
    
    # Inject master assumptions
    ws_is[f"{col}12"].value, ws_is[f"{col}12"].font = assump_rev_growth, blue_font
    ws_is[f"{col}13"].value, ws_is[f"{col}13"].font = assump_gross_margin, blue_font
    ws_is[f"{col}14"].value, ws_is[f"{col}14"].font = assump_sga_margin, blue_font
    ws_is[f"{col}15"].value, ws_is[f"{col}15"].font = assump_tax_rate, blue_font

    # Total Revenue
    ws_is.cell(row=2, column=col_idx, value=f"={prev}2*(1+{col}12)").font = black_font 
    # Cost of Revenue
    ws_is.cell(row=3, column=col_idx, value=f"={col}2*(1-{col}13)").font = black_font 
    # Gross Profit
    ws_is.cell(row=4, column=col_idx, value=f"={col}2-{col}3").font = black_font      
    # SG&A
    ws_is.cell(row=5, column=col_idx, value=f"={col}2*{col}14").font = black_font     
    # Operating Income (EBIT)
    ws_is.cell(row=6, column=col_idx, value=f"={col}4-{col}5").font = black_font      
    # Interest Expense (Prevents Circular Reference)
    ws_is.cell(row=7, column=col_idx, value=f"='Debt Schedule'!{col}14*'Debt Schedule'!{col}4").font = green_font 
    # Tax Provision
    ws_is.cell(row=8, column=col_idx, value=f"=({col}6-{col}7)*{col}15").font = black_font 
    # Net Income
    ws_is.cell(row=9, column=col_idx, value=f"={col}6-{col}7-{col}8").font = black_font 

## Step 3: The Balance Sheet

The Balance Sheet calculates capital requirements based on operating efficiency. Cash is intentionally omitted from the forecast logic here—it acts as the ultimate balancing plug derived from the final Cash Flow Statement.

### Formulas Used:
*   **Receivables:** Income Statement Revenue * Receivables %
*   **Inventory:** Income Statement COGS * Inventory %
*   **Accounts Payable:** Income Statement COGS * Payables %
*   **Net PPE:** Prior Net PPE - CapEx (from CFS) - Depreciation
*   **Retained Earnings:** Prior RE + Net Income - Dividends Paid
*   **Treasury Stock:** Prior Treasury Stock - Share Repurchases
*   **Historical Plug:** Total Captured Assets - Total Captured Liabilities/Equity

In [ ]:
ws_bs = wb.create_sheet("Balance Sheet")
write_clean_df(ws_bs, clean_bs)
for i, year in enumerate(forecast_cols, start=7): ws_bs.cell(row=1, column=i, value=year).font = bold_font

ws_bs["A12"] = "Other Net Historicals (Plug)"
for col_idx in range(2, 7):
    col = get_column_letter(col_idx)
    # Historical Plug = Captured Assets - Captured Liab/Eq
    ws_bs.cell(row=12, column=col_idx, value=f"=SUM({col}2:{col}5)-(SUM({col}6:{col}10)-{col}11)").font = black_font

ws_bs["A14"] = "WORKING CAPITAL ASSUMPTIONS"; ws_bs["A14"].font = bold_font
ws_bs["A15"] = "Receivables (% of Revenue)"
ws_bs["A16"] = "Inventory (% of COGS)"
ws_bs["A17"] = "Payables (% of COGS)"

for col_idx in range(7, 12):
    col = get_column_letter(col_idx)
    prev = get_column_letter(col_idx - 1)
    
    ws_bs[f"{col}15"].value, ws_bs[f"{col}15"].font = 0.015, blue_font
    ws_bs[f"{col}16"].value, ws_bs[f"{col}16"].font = 0.25, blue_font
    ws_bs[f"{col}17"].value, ws_bs[f"{col}17"].font = 0.12, blue_font
    
    # Cash = Prior Cash + Net Change in Cash
    ws_bs.cell(row=2, column=col_idx, value=f"={prev}2+'Cash Flow Statement'!{col}12").font = green_font 
    # Receivables
    ws_bs.cell(row=3, column=col_idx, value=f"='Income Statement'!{col}2*{col}15").font = green_font      
    # Inventory
    ws_bs.cell(row=4, column=col_idx, value=f"='Income Statement'!{col}3*{col}16").font = green_font      
    # Net PPE
    ws_bs.cell(row=5, column=col_idx, value=f"={prev}5-'Cash Flow Statement'!{col}6-'Cash Flow Statement'!{col}3").font = green_font
    # Accounts Payable
    ws_bs.cell(row=6, column=col_idx, value=f"='Income Statement'!{col}3*{col}17").font = green_font      
    # Accrued Expenses
    ws_bs.cell(row=7, column=col_idx, value=f"={prev}7").font = black_font                                
    # Long Term Debt
    ws_bs.cell(row=8, column=col_idx, value=f"='Debt Schedule'!{col}17").font = green_font                                
    # Common Stock
    ws_bs.cell(row=9, column=col_idx, value=f"={prev}9").font = black_font                                
    # Retained Earnings
    ws_bs.cell(row=10, column=col_idx, value=f"={prev}10+'Income Statement'!{col}9+'Cash Flow Statement'!{col}9").font = green_font 
    # Treasury Stock
    ws_bs.cell(row=11, column=col_idx, value=f"={prev}11-'Cash Flow Statement'!{col}10").font = green_font 
    # Historical Plug
    ws_bs.cell(row=12, column=col_idx, value=f"={prev}12").font = black_font 

## Step 4: The Cash Flow Statement

The Cash Flow Statement bridges accounting profit (Net Income) to actual cash entering or leaving the bank account.

### Formulas Used:
*   **Depreciation (D&A):** Revenue * D&A Margin
*   **Change in Working Capital:** (Prior Receivables - Current Receivables) + (Prior Inventory - Current Inventory) + (Current Payables - Prior Payables)
*   **Operating Cash Flow (CFO):** Net Income + D&A + Change in Working Capital
*   **Capital Expenditures:** Revenue * CapEx Margin (Output as a negative cash outflow)
*   **Net Change in Cash:** CFO + CFI + CFF

In [ ]:
ws_cf = wb.create_sheet("Cash Flow Statement")
write_clean_df(ws_cf, clean_cf)
for i, year in enumerate(forecast_cols, start=7): ws_cf.cell(row=1, column=i, value=year).font = bold_font

ws_cf["A14"] = "CASH FLOW ASSUMPTIONS"; ws_cf["A14"].font = bold_font
ws_cf["A15"] = "D&A (% of Revenue)"
ws_cf["A16"] = "CapEx (% of Revenue)"

for col_idx in range(7, 12):
    col = get_column_letter(col_idx)
    prev = get_column_letter(col_idx - 1)
    
    ws_cf[f"{col}15"].value, ws_cf[f"{col}15"].font = 0.025, blue_font
    ws_cf[f"{col}16"].value, ws_cf[f"{col}16"].font = assump_capex_margin, blue_font
    
    # Net Income
    ws_cf.cell(row=2, column=col_idx, value=f"='Income Statement'!{col}9").font = green_font 
    # D&A
    ws_cf.cell(row=3, column=col_idx, value=f"='Income Statement'!{col}2*{col}15").font = green_font 
    # Change in WC 
    ws_cf.cell(row=4, column=col_idx, value=f"=('Balance Sheet'!{prev}3-'Balance Sheet'!{col}3)+('Balance Sheet'!{prev}4-'Balance Sheet'!{col}4)+('Balance Sheet'!{col}6-'Balance Sheet'!{prev}6)").font = green_font
    # Operating Cash Flow (CFO)
    ws_cf.cell(row=5, column=col_idx, value=f"=SUM({col}2:{col}4)").font = black_font 
    # CapEx 
    ws_cf.cell(row=6, column=col_idx, value=f"='Income Statement'!{col}2*{col}16").font = green_font 
    # Investing Cash Flow (CFI) 
    ws_cf.cell(row=7, column=col_idx, value=f"={col}6").font = black_font 
    # Repayment of Debt
    ws_cf.cell(row=8, column=col_idx, value=f"=-('Debt Schedule'!{col}15+'Debt Schedule'!{col}16)").font = green_font 
    # Dividends & Repurchases
    ws_cf.cell(row=9, column=col_idx, value=-500000000).font = blue_font 
    ws_cf.cell(row=10, column=col_idx, value=-600000000).font = blue_font 
    # Financing Cash Flow (CFF)
    ws_cf.cell(row=11, column=col_idx, value=f"=SUM({col}8:{col}10)").font = black_font 
    # Change in Cash
    ws_cf.cell(row=12, column=col_idx, value=f"={col}5+{col}7+{col}11").font = black_font 

## Step 5: The Balance Sheet Check

A financial model is useless if it doesn't balance. We build an explicit audit row at the bottom of the Balance Sheet that constantly verifies $Assets - Liabilities - Equity = 0$.

### Formulas Used:
*   **Check Formula:** `=IF(ROUND(Assets - (Liabilities + Equity), 0) = 0, "OK - BALANCED", "ERROR: IMBALANCE")`

In [ ]:
ws_bs["A20"] = "MODEL AUDIT CHECK"; ws_bs["A20"].font = bold_font

for col_idx in range(7, 12):
    col = get_column_letter(col_idx)
    
    # Sum(Assets) - (Sum(Liab/Eq) - TreasuryStock + HistoricalPlug)
    asset_sum = f"SUM({col}2:{col}5)"
    liab_eq_sum = f"(SUM({col}6:{col}10)-{col}11+{col}12)"
    check_math = f"{asset_sum}-{liab_eq_sum}"
    
    # Excel IF statement
    if_formula = f'=IF(ROUND({check_math}, 0)=0, "OK - BALANCED", "ERROR: " & {check_math})'
    
    check_cell = ws_bs.cell(row=20, column=col_idx, value=if_formula)
    check_cell.font = bold_font
    check_cell.fill = yellow_fill

## Step 6: The Debt Schedule (Cash Sweep)

This schedule calculates Cash Available for Debt Service (CADS). If the company has excess cash beyond its minimum threshold, it dynamically "sweeps" that cash to pay down principal debt.

### Formulas Used:
*   **CADS:** Beginning Cash + CFO + CFI + Dividends & Repurchases
*   **Cash Sweep:** MIN(Beginning Debt, MAX(0, CADS - Minimum Cash Balance))
*   **Ending Debt Balance:** Beginning Debt - Mandatory Repayment - Cash Sweep

In [ ]:
ws_debt = wb.create_sheet("Debt Schedule")
ws_debt["A1"] = "DEBT SCHEDULE"; ws_debt["A1"].font = bold_font
for i, year in enumerate(forecast_cols, start=7): ws_debt.cell(row=1, column=i, value=year).font = bold_font

ws_debt["A2"] = "ASSUMPTIONS"; ws_debt["A2"].font = bold_font
ws_debt["A3"] = "Minimum Cash Balance"
ws_debt["A4"] = "Interest Rate on Debt"

ws_debt["A6"] = "CASH AVAILABLE FOR DEBT SERVICE (CADS)"; ws_debt["A6"].font = bold_font
ws_debt["A7"] = "Beginning Cash Balance"
ws_debt["A8"] = "Operating Cash Flow"
ws_debt["A9"] = "Investing Cash Flow"
ws_debt["A10"] = "Dividends & Repurchases"
ws_debt["A11"] = "CADS"

ws_debt["A13"] = "LONG TERM DEBT ROLL-FORWARD"; ws_debt["A13"].font = bold_font
ws_debt["A14"] = "Beginning Debt Balance"
ws_debt["A15"] = "Less: Mandatory Repayment"
ws_debt["A16"] = "Less: Cash Sweep (Optional)"
ws_debt["A17"] = "Ending Debt Balance"

for col_idx in range(7, 12):
    col = get_column_letter(col_idx)
    prev = get_column_letter(col_idx - 1)
    
    ws_debt[f"{col}3"].value, ws_debt[f"{col}3"].font = 200000000, blue_font
    ws_debt[f"{col}4"].value, ws_debt[f"{col}4"].font = 0.045, blue_font
    
    # CADS inputs
    ws_debt.cell(row=7, column=col_idx, value=f"='Balance Sheet'!{prev}2").font = green_font
    ws_debt.cell(row=8, column=col_idx, value=f"='Cash Flow Statement'!{col}5").font = green_font
    ws_debt.cell(row=9, column=col_idx, value=f"='Cash Flow Statement'!{col}7").font = green_font
    ws_debt.cell(row=10, column=col_idx, value=f"='Cash Flow Statement'!{col}9+'Cash Flow Statement'!{col}10").font = green_font
    ws_debt.cell(row=11, column=col_idx, value=f"=SUM({col}7:{col}10)").font = black_font
    
    # Beginning Debt Balance
    if col_idx == 7:
        ws_debt.cell(row=14, column=col_idx, value=f"='Balance Sheet'!F8").font = green_font
    else:
        ws_debt.cell(row=14, column=col_idx, value=f"={prev}17").font = black_font
        
    # Mandatory Repayment
    ws_debt.cell(row=15, column=col_idx, value=0).font = blue_font 
    # Cash Sweep 
    ws_debt.cell(row=16, column=col_idx, value=f"=MIN({col}14, MAX(0, {col}11-{col}3))").font = black_font
    # Ending Debt Balance
    ws_debt.cell(row=17, column=col_idx, value=f"={col}14-{col}15-{col}16").font = black_font

## Step 7: Advanced DCF Valuation & Sensitivity Analysis

A rigorous DCF derives its discount rate directly from the market, relies on multiple valuation approaches, and tests for vulnerability via sensitivity analysis.

### WACC Derivation (CAPM)
We avoid hardcoding generic WACC rates by calculating the true Market Value of Equity ($E = Price \times Shares$) to derive objective weightings.
$$K_e = R_f + \beta \cdot ERP$$
$$WACC = \frac{E}{V} \cdot K_e + \frac{D}{V} \cdot K_d \cdot (1 - t)$$

### Valuation Cross-Check
We calculate two terminal methodologies and average the outputs:
1.  **Gordon Growth Method:** Values perpetual cash flow growth.
2.  **Exit Multiple Method:** Values the terminal year based on an EV/EBITDA multiple.

### Sensitivity Grid
We generate a robust, formula-driven $5 \times 5$ matrix to stress-test the implied share price across fluctuating WACC and Growth combinations.

In [ ]:
ws_dcf = wb.create_sheet("DCF Valuation")
ws_dcf["A1"] = "DISCOUNTED CASH FLOW VALUATION"; ws_dcf["A1"].font = bold_font
for i, year in enumerate(forecast_cols, start=7): ws_dcf.cell(row=1, column=i, value=year).font = bold_font

ws_dcf["A3"] = "WACC DERIVATION (CAPM)"; ws_dcf["A3"].font = bold_font
ws_dcf["A4"] = "Risk Free Rate (10-Yr Treasury)"
ws_dcf["A5"] = "Company Beta (Damodaran Industry Proxy)"
ws_dcf["A6"] = "Equity Risk Premium"
ws_dcf["A7"] = "Cost of Equity (Ke)"
ws_dcf["A8"] = "After-Tax Cost of Debt (Kd)"
ws_dcf["A9"] = "Current Share Price (Live)"
ws_dcf["A10"] = "Shares Outstanding (Raw)"
ws_dcf["A11"] = "Market Value of Equity (E)"
ws_dcf["A12"] = "Market Value of Debt (D)"
ws_dcf["A13"] = "Total Capital (V)"
ws_dcf["A14"] = "Weight of Equity (E/V)"
ws_dcf["A15"] = "Weight of Debt (D/V)"
ws_dcf["A16"] = "Calculated WACC"; ws_dcf["A16"].font = bold_font
ws_dcf["A17"] = "Terminal Growth Rate"
ws_dcf["A18"] = "Exit Multiple (EV/EBITDA)"
ws_dcf["A19"] = "Discount Period"

# WACC Inputs & Derivations
ws_dcf["G4"].value, ws_dcf["G4"].font = risk_free_rate, blue_font
ws_dcf["G5"].value, ws_dcf["G5"].font = assump_beta, blue_font
ws_dcf["G6"].value, ws_dcf["G6"].font = assump_erp, blue_font
ws_dcf["G7"].value = "=G4+(G5*G6)"; ws_dcf["G7"].font = black_font
ws_dcf["G8"].value = "='Debt Schedule'!G4*(1-'Income Statement'!G15)"; ws_dcf["G8"].font = green_font
ws_dcf["G9"].value, ws_dcf["G9"].font = live_price, blue_font
ws_dcf["G10"].value, ws_dcf["G10"].font = assump_shares_out, blue_font
ws_dcf["G11"].value = "=G9*G10"; ws_dcf["G11"].font = black_font
ws_dcf["G12"].value = "='Balance Sheet'!F8"; ws_dcf["G12"].font = green_font
ws_dcf["G13"].value = "=G11+G12"; ws_dcf["G13"].font = black_font
ws_dcf["G14"].value = "=G11/G13"; ws_dcf["G14"].font = black_font
ws_dcf["G15"].value = "=G12/G13"; ws_dcf["G15"].font = black_font
ws_dcf["G16"].value = "=(G14*G7)+(G15*G8)"; ws_dcf["G16"].font = bold_font
ws_dcf["G16"].fill = blue_fill

ws_dcf["G17"].value, ws_dcf["G17"].font = assump_tgr, blue_font   
ws_dcf["G18"].value, ws_dcf["G18"].font = assump_exit_mult, blue_font  

for col_idx in range(7, 12): ws_dcf.cell(row=19, column=col_idx, value=col_idx-6).font = blue_font 

ws_dcf["A21"] = "UNLEVERED FREE CASH FLOW"; ws_dcf["A21"].font = bold_font
ws_dcf["A22"] = "EBIT"
ws_dcf["A23"] = "Less: Taxes on EBIT"
ws_dcf["A24"] = "Net Operating Profit After Tax (NOPAT)"
ws_dcf["A25"] = "Plus: D&A"
ws_dcf["A26"] = "Less: Capital Expenditures"
ws_dcf["A27"] = "Less: Change in Working Capital"
ws_dcf["A28"] = "Unlevered Free Cash Flow (UFCF)"

for col_idx in range(7, 12):
    col = get_column_letter(col_idx)
    ws_dcf.cell(row=22, column=col_idx, value=f"='Income Statement'!{col}6").font = green_font 
    ws_dcf.cell(row=23, column=col_idx, value=f"={col}22*'Income Statement'!{col}15").font = green_font 
    ws_dcf.cell(row=24, column=col_idx, value=f"={col}22-{col}23").font = black_font 
    ws_dcf.cell(row=25, column=col_idx, value=f"='Cash Flow Statement'!{col}3").font = green_font 
    ws_dcf.cell(row=26, column=col_idx, value=f"='Cash Flow Statement'!{col}6").font = green_font 
    ws_dcf.cell(row=27, column=col_idx, value=f"='Cash Flow Statement'!{col}4").font = green_font
    ws_dcf.cell(row=28, column=col_idx, value=f"={col}24+{col}25+{col}26+{col}27").font = black_font

ws_dcf["A30"] = "PRESENT VALUE OF CASH FLOWS"; ws_dcf["A30"].font = bold_font
ws_dcf["A31"] = "Discount Factor"
ws_dcf["A32"] = "Present Value of UFCF"

for col_idx in range(7, 12):
    col = get_column_letter(col_idx)
    ws_dcf.cell(row=31, column=col_idx, value=f"=(1+$G$16)^{col}19").font = black_font
    ws_dcf.cell(row=32, column=col_idx, value=f"={col}28/{col}31").font = black_font

# Approach 1: Perpetuity Growth
ws_dcf["A34"] = "TERMINAL VALUE - GORDON GROWTH"; ws_dcf["A34"].font = bold_font
ws_dcf["A35"] = "Terminal Value"
ws_dcf["A36"] = "PV of Terminal Value"
ws_dcf["A37"] = "PV of Stage 1 Cash Flows"
ws_dcf["A38"] = "Enterprise Value"
ws_dcf["A39"] = "Plus: Current Cash"
ws_dcf["A40"] = "Less: Current Debt"
ws_dcf["A41"] = "Equity Value"
ws_dcf["A42"] = "Implied Price (Perpetuity Method)"; ws_dcf["A42"].font = bold_font

ws_dcf["C35"].value = "=(K28*(1+G17))/(G16-G17)"; ws_dcf["C35"].font = black_font
ws_dcf["C36"].value = "=C35/K31"; ws_dcf["C36"].font = black_font
ws_dcf["C37"].value = "=SUM(G32:K32)"; ws_dcf["C37"].font = black_font
ws_dcf["C38"].value = "=C36+C37"; ws_dcf["C38"].font = black_font
ws_dcf["C39"].value = "='Balance Sheet'!F2"; ws_dcf["C39"].font = green_font
ws_dcf["C40"].value = "='Balance Sheet'!F8"; ws_dcf["C40"].font = green_font
ws_dcf["C41"].value = "=C38+C39-C40"; ws_dcf["C41"].font = black_font
ws_dcf["C42"].value = "=C41/G10"; ws_dcf["C42"].font = bold_font

# Approach 2: Exit Multiple
ws_dcf["A44"] = "TERMINAL VALUE - EXIT MULTIPLE"; ws_dcf["A44"].font = bold_font
ws_dcf["A45"] = "Final Year EBITDA"
ws_dcf["A46"] = "Terminal Value"
ws_dcf["A47"] = "PV of Terminal Value"
ws_dcf["A48"] = "Enterprise Value"
ws_dcf["A49"] = "Plus: Current Cash"
ws_dcf["A50"] = "Less: Current Debt"
ws_dcf["A51"] = "Equity Value"
ws_dcf["A52"] = "Implied Price (Multiple Method)"; ws_dcf["A52"].font = bold_font

ws_dcf["C45"].value = "=K22+K25"; ws_dcf["C45"].font = black_font
ws_dcf["C46"].value = "=C45*G18"; ws_dcf["C46"].font = black_font
ws_dcf["C47"].value = "=C46/K31"; ws_dcf["C47"].font = black_font
ws_dcf["C48"].value = "=C37+C47"; ws_dcf["C48"].font = black_font
ws_dcf["C49"].value = "=C39"; ws_dcf["C49"].font = black_font
ws_dcf["C50"].value = "=C40"; ws_dcf["C50"].font = black_font
ws_dcf["C51"].value = "=C48+C49-C50"; ws_dcf["C51"].font = black_font
ws_dcf["C52"].value = "=C51/G10"; ws_dcf["C52"].font = bold_font

# Blended Output
ws_dcf["A54"] = "BLENDED TARGET PRICE (50/50)"; ws_dcf["A54"].font = bold_font
ws_dcf["C55"].value = "=AVERAGE(C42, C52)"; ws_dcf["C55"].font = bold_font
ws_dcf["C55"].fill = green_fill

# ==========================================
# SENSITIVITY GRID
# ==========================================
ws_dcf["A58"] = "SENSITIVITY ANALYSIS: WACC vs. TGR"; ws_dcf["A58"].font = bold_font
ws_dcf["B59"] = "WACC \\ TGR"

# Define row inputs (TGR) and column inputs (WACC)
tgr_offsets = [-0.005, -0.0025, 0, 0.0025, 0.005]
wacc_offsets = [-0.01, -0.005, 0, 0.005, 0.01]

for r_idx, t_off in enumerate(tgr_offsets, start=60):
    ws_dcf.cell(row=r_idx, column=2, value=f"=$G$17+{t_off}").font = bold_font
    
for c_idx, w_off in enumerate(wacc_offsets, start=3):
    col = get_column_letter(c_idx)
    ws_dcf.cell(row=59, column=c_idx, value=f"=$G$16+{w_off}").font = bold_font
    
    for r_idx in range(60, 65):
        # Matrix Math: Evaluates the Gordon Growth logic dynamically for every grid intersection
        formula = f"=($C$37 + ((K$28*(1+$B{r_idx}))/({col}$59-$B{r_idx}))/(1+{col}$59)^5 + $C$39 - $C$40)/$G$10"
        cell = ws_dcf.cell(row=r_idx, column=c_idx, value=formula)
        cell.number_format = '"$"#,##0.00'

## Step 8: Executive Summary & Dynamic Recommendation

The Executive Summary drives the narrative. Hardcoding "BUY" or "SELL" in a model is a critical error. Instead, we fetch the live market price and link it to an Excel `=IF()` statement that evaluates the target upside dynamically.

**Important Excel Note:** `openpyxl` writes raw formulas to the workbook. When you open the downloaded `.xlsx` file, **you must press F9 or click 'Enable Editing'** to force Excel's engine to calculate and populate the cached values.

In [ ]:
ws_exec = wb.active
ws_exec.title = "Executive Summary"

ws_exec["A1"] = "Tractor Supply Company (TSCO) - Financial Analysis"
ws_exec["A1"].font = title_font

# Inject the Live Market Price
ws_exec["A3"] = "Current Market Price (Live):"; ws_exec["A3"].font = bold_font
ws_exec["B3"].value = live_price
ws_exec["B3"].font = blue_font
ws_exec["B3"].number_format = '"$"#,##0.00'

ws_exec["A4"] = "Implied Target Price (Blended DCF):"; ws_exec["A4"].font = bold_font
ws_exec["B4"].value = "='DCF Valuation'!C55"
ws_exec["B4"].font = green_font
ws_exec["B4"].number_format = '"$"#,##0.00'

# Dynamic Recommendation Logic
ws_exec["A6"] = "Recommendation:"; ws_exec["A6"].font = bold_font
recommendation_formula = '=IF(B4 > B3*1.1, "STRONG BUY", IF(B4 > B3, "BUY", IF(B4 < B3*0.9, "SELL", "HOLD")))'
ws_exec["B6"].value = recommendation_formula
ws_exec["B6"].font = bold_font
ws_exec["B6"].fill = yellow_fill

ws_exec["A8"] = "Key Investment Drivers:"; ws_exec["A8"].font = bold_font
drivers = [
    f"1. Top-Line Growth: Revenue projected at a conservative {assump_rev_growth*100:.1f}% CAGR through 2030E.",
    f"2. Margin Stability: Gross margins modeled flat at {assump_gross_margin*100:.1f}% with SG&A optimized at {assump_sga_margin*100:.1f}%.",
    f"3. Valuation Derivation: Utilized CAPM to structure a dynamic WACC, cross-checked with a {assump_exit_mult}x EV/EBITDA exit multiple.",
]
for idx, driver in enumerate(drivers, start=9):
    ws_exec.cell(row=idx, column=1, value=driver)
    
ws_exec["A13"] = "Simplifying Assumptions & Limitations:"; ws_exec["A13"].font = bold_font
assumptions = [
    "1. Share Count vs. Buybacks: Modeled $600M annual share repurchases as a cash outflow, but held shares outstanding constant for valuation to avoid artificial EPS accretion.",
    "2. Aggressive Deleverage: Modeled a 100% cash sweep for balances over $200M, preventing cash accumulation on the Balance Sheet in favor of zero-debt."
]
for idx, note in enumerate(assumptions, start=14):
    ws_exec.cell(row=idx, column=1, value=note)

# Clean up column widths globally
for sheet in wb.sheetnames:
    wb[sheet].column_dimensions['A'].width = 40

## Step 9: Export the Deliverable

We save the memory workbook to a `.xlsx` file and trigger the browser download so it can be attached to the application email.

⚠️ **CRITICAL INSTRUCTION FOR THE USER:** ⚠️
Because `openpyxl` writes formula strings, **this workbook will appear blank or broken when first downloaded.** You MUST open the downloaded file in Microsoft Excel, click **"Enable Editing"** (or press `F9`), and allow the Excel calculation engine to cache the math. Save the file manually before sending it to any reviewers.

In [ ]:
file_name = "TSCO_Final_IB_Model.xlsx"
wb.save(file_name)
print(f"Model generated successfully as {file_name}")
print("\nIMPORTANT: Open the downloaded file in Excel and hit 'Enable Editing' to calculate the formulas before sending!")

# Trigger Colab Download
files.download(file_name)